# Βήμα 2 – Προεπεξεργασία UNSW-NB15 (Σύστημα Α)
Μετατρέπουμε τα δεδομένα σε μορφή που μπορεί να «φάει» το νευρωνικό δίκτυο:
αριθμοί, στην ίδια κλίμακα, χωρίς κείμενο. **Όλα τα «μαθαίνω από τα δεδομένα» (scaler, encoder) γίνονται μόνο στο train set.**

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

DATA = "../data/"
MODELS = "../models/"
RANDOM_STATE = 42          # για αναπαραγώγιμα αποτελέσματα

## 2.1 Φόρτωση και ορισμός στόχων
- `label` → επίθεση (1) ή κανονική ροή (0)
- `attack_cat` → τύπος (Normal + 9 τύποι επίθεσης)
- `severity` → σοβαρότητα, την ορίζουμε εμείς από τον τύπο (δεν υπάρχει στο dataset)

In [ ]:
train_full = pd.read_csv(DATA + "UNSW_NB15_train-set.csv")
test = pd.read_csv(DATA + "UNSW_NB15_test-set.csv")

SEVERITY_MAP = {
    "Normal": "None",
    "Reconnaissance": "Low", "Analysis": "Low", "Fuzzers": "Low",
    "DoS": "Medium", "Generic": "Medium",
    "Exploits": "High", "Backdoor": "High", "Shellcode": "High", "Worms": "High",
}
SEVERITY_WEIGHT = {"None": 0.0, "Low": 0.33, "Medium": 0.66, "High": 1.0}

for df in (train_full, test):
    df["attack_cat"] = df["attack_cat"].str.strip()
    df["severity"] = df["attack_cat"].map(SEVERITY_MAP)

print(train_full.shape, test.shape)
print("Κενά στη σοβαρότητα:", train_full["severity"].isna().sum(), test["severity"].isna().sum())
print(pd.crosstab(train_full["attack_cat"], train_full["severity"]))

## 2.2 Διπλότυπες γραμμές
Το UNSW-NB15 έχει πολλές **ίδιες** γραμμές (κυρίως τύπου Generic). Αν μείνουν, η ίδια ροή μπορεί να βρεθεί και στο train και στο validation,
και το validation θα δείχνει ψεύτικα καλά αποτελέσματα. Τις αφαιρούμε **μόνο από το train**· το test μένει ως έχει (επίσημο).

In [ ]:
DROP_DUPLICATES = True

before = len(train_full)
if DROP_DUPLICATES:
    train_full = train_full.drop_duplicates(subset=[c for c in train_full.columns if c != "id"])
print(f"Γραμμές: {before} → {len(train_full)} (αφαιρέθηκαν {before - len(train_full)})")
print(train_full["attack_cat"].value_counts())

## 2.3 Είσοδοι
- **Κατηγορικές** (κείμενο): `proto`, `service`, `state` → one-hot encoding
- **Αριθμητικές**: όλες οι υπόλοιπες. Όσες έχουν πολύ «απλωμένη» κατανομή (skewness > 2, π.χ. bytes) περνούν πρώτα από `log(1+x)`.
- Το `id` είναι απλώς αρίθμηση, δεν είναι χαρακτηριστικό → το πετάμε.

In [ ]:
TARGETS = ["label", "attack_cat", "severity"]
CAT_COLS = ["proto", "service", "state"]
NUM_COLS = [c for c in train_full.columns if c not in TARGETS + CAT_COLS + ["id"]]

skew = train_full[NUM_COLS].skew()
LOG_COLS = skew[skew > 2].index.tolist()
PLAIN_COLS = [c for c in NUM_COLS if c not in LOG_COLS]

print(f"{len(NUM_COLS)} αριθμητικές ({len(LOG_COLS)} με log), {len(CAT_COLS)} κατηγορικές")
print("Με log:", LOG_COLS)

## 2.4 Διαχωρισμός train / validation (85% / 15%)
Με `stratify` ώστε κάθε τύπος επίθεσης να υπάρχει στην ίδια αναλογία και στα δύο κομμάτια (σημαντικό για τους σπάνιους, π.χ. Worms).

In [ ]:
train_df, val_df = train_test_split(
    train_full, test_size=0.15, stratify=train_full["attack_cat"], random_state=RANDOM_STATE
)
print("train:", train_df.shape, "| val:", val_df.shape, "| test:", test.shape)

## 2.5 Μετασχηματισμός εισόδων
- `OneHotEncoder(min_frequency=50)`: τιμές που εμφανίζονται < 50 φορές (π.χ. σπάνια πρωτόκολλα) μπαίνουν σε μία κοινή στήλη «infrequent»· άγνωστες τιμές στο test δεν σπάνε τον κώδικα.
- `StandardScaler`: κάθε αριθμητική στήλη → μέσος 0, τυπική απόκλιση 1.

In [ ]:
preprocessor = ColumnTransformer([
    ("log", Pipeline([
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ]), LOG_COLS),
    ("num", StandardScaler(), PLAIN_COLS),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50,
                          sparse_output=False), CAT_COLS),
])

X_train = preprocessor.fit_transform(train_df).astype("float32")   # fit ΜΟΝΟ στο train
X_val = preprocessor.transform(val_df).astype("float32")
X_test = preprocessor.transform(test).astype("float32")
feature_names = preprocessor.get_feature_names_out().tolist()

print("Σχήματα:", X_train.shape, X_val.shape, X_test.shape)
print("Train: μέσος ≈ 0 →", X_train[:, :len(NUM_COLS)].mean().round(3),
      "| τυπ. απόκλιση ≈ 1 →", X_train[:, :len(NUM_COLS)].std().round(3))

## 2.6 Κωδικοποίηση στόχων σε αριθμούς

In [ ]:
ATTACK_CLASSES = ["Normal", "Generic", "Exploits", "Fuzzers", "DoS",
                  "Reconnaissance", "Analysis", "Backdoor", "Shellcode", "Worms"]
SEVERITY_CLASSES = ["None", "Low", "Medium", "High"]

def encode_targets(df):
    return {
        "label": df["label"].to_numpy("int8"),
        "attack_cat": df["attack_cat"].map({c: i for i, c in enumerate(ATTACK_CLASSES)}).to_numpy("int8"),
        "severity": df["severity"].map({c: i for i, c in enumerate(SEVERITY_CLASSES)}).to_numpy("int8"),
    }

y_train, y_val, y_test = encode_targets(train_df), encode_targets(val_df), encode_targets(test)
print({k: np.bincount(v) for k, v in y_train.items()})

## 2.7 Αποθήκευση
Αποθηκεύουμε τα έτοιμα δεδομένα, τον preprocessor (για να μετασχηματίζουμε **νέες** ροές με τον ίδιο ακριβώς τρόπο) και τις ρυθμίσεις.

In [ ]:
np.savez_compressed(
    DATA + "unsw_prepared.npz",
    X_train=X_train, X_val=X_val, X_test=X_test,
    **{f"y_train_{k}": v for k, v in y_train.items()},
    **{f"y_val_{k}": v for k, v in y_val.items()},
    **{f"y_test_{k}": v for k, v in y_test.items()},
)
joblib.dump(preprocessor, MODELS + "unsw_preprocessor.joblib")

meta = {
    "cat_cols": CAT_COLS, "num_cols": NUM_COLS, "log_cols": LOG_COLS,
    "feature_names": feature_names,
    "attack_classes": ATTACK_CLASSES, "severity_classes": SEVERITY_CLASSES,
    "severity_map": SEVERITY_MAP, "severity_weight": SEVERITY_WEIGHT,
    "drop_duplicates": DROP_DUPLICATES, "random_state": RANDOM_STATE,
}
with open(MODELS + "unsw_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Αποθηκεύτηκαν: data/unsw_prepared.npz, models/unsw_preprocessor.joblib, models/unsw_meta.json")